# Single-shot IQ scatter — 10,000 readout shots at 2.75 GHz

Record **10,000 single-shot IQ acquisitions** at a fixed readout frequency and plot them as a
scatter. The **readout drive** channel (ch 1) plays a measurement tone; the **demod** channel
(ch 2) demodulates the looped-back signal, returning one complex integral `(I, Q)` per shot. On a
real converter each shot carries independent thermal + quantization noise, so the 10,000 points
form a blob whose spread is the readout noise floor.

This is a **hardware** notebook — it drives a real ZCU216 with the `xm650-loopback` build over
`RemoteDriver`, the same as [`vna.ipynb`](vna.ipynb) / [`remote_pulse.ipynb`](remote_pulse.ipynb),
so it is *not* executed in CI (a co-sim loopback is deterministic and would collapse every shot to
one point). Server setup: [docs/software/board-server.md](../docs/software/board-server.md).

The unified core RAM is 16 KB (4096 words), so a 10,000-shot buffer (20,000 words) can't live
on-core at once. We fire **1,000 shots per run** (a 2,000-word `out` buffer) and collect
**10 reruns** — one `rq.setup` loads the image, then each `rq.rerun` reuses it and returns another
1,000 shots.

## Connect to the board

In [ ]:
import time

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

from riscq import run as rq
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, ParamTable, compile_kernel, kernel
from riscq.map import LEAD, READOUT_LEAD, SocMap, SocParams
from riscq.pulses import Pulse, envelopes, units

BOARD = "192.168.1.122"                   # the ZCU216's LAN address (or the full PYRO: uri)

drv = RemoteDriver(BOARD, 9091)
print("server:", drv.board.info())
print("bundles on the board:", drv.board.bundles())

# first time only — push the build up and load it (~100 MB, a minute on GbE):
# upload_bundle(drv, "xm650-loopback",
#               xsa="../build/xm650-loopback/top.xsa",                   # write_hw_platform export
#               params_json="../software/configs/xm650-loopback.json")   # the SAME JSON the build used
# info = drv.board.load("xm650-loopback")                                # full RF bring-up; returns info()
# print(info)
# assert info["mts_result"] == 0, "multi-tile sync missed its target latencies"

m = SocMap(SocParams.from_json(drv.board.get_params()))   # always matches the loaded bitstream
fs = units.sample_rate(m.params)
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{fs / 1e9:.0f} GS/s DACs, first Nyquist zone 0 - {fs / 2e9:.1f} GHz")

## The kernel

One on-core kernel fires `shots` acquisitions on a fixed time grid and stores every shot's raw IQ
integral into `out`. The readout drive and demod carrier are tuned once from host-computed codes:
`freq_to_code` for the DAC-rate drive and `demod_freq_to_code` for the ADC-rate demod (which is 4×
the drive code — the ADC has 4 samples/batch to the DAC's 16 — and folds mod 2^16 above its own
Nyquist).

In [ ]:
@kernel
def iq_shots(ro: ParamTable, demod: ParamTable, out: Array, shots: int, period: int,
             ro_code: int, demod_code: int):
    """One readout tone at a fixed frequency, `shots` single-shot acquisitions, every shot's raw
    IQ integral stored into `out` (2*shots words). The readout drive (ch 1) plays the measurement
    tone; the demod carrier (ch 2) demodulates the looped-back signal into one (I, Q) per shot."""
    init_pulse_params(ro.pulses)               # noqa: F821  load the readout-drive slot
    init_pulse_params(demod.pulses)            # noqa: F821  load the demod-carrier slot
    set_freq(ro, ro_code)                      # noqa: F821  DAC-rate readout drive
    set_freq(demod, demod_code)                # noqa: F821  ADC-rate demod carrier (demod_freq_to_code)
    t = now() + period                         # noqa: F821  first grid slot (retune lands >> LEAD ahead)
    k = 0
    for s in range(shots):
        play(ro, ro["meas"], t)                # noqa: F821  drive covers the demod window
        play(demod, demod["sq"], t)            # noqa: F821  firing the demod IS the readout
        wait_until(t + READOUT_LEAD)            # noqa: F821  let the stale result level drop
        read_res()                             # noqa: F821  HALT until this shot's integral settles
        out[k] = read_real()                   # noqa: F821  this shot's I
        out[k + 1] = read_imag()               # noqa: F821  this shot's Q
        k = k + 2
        t = t + period                         # noqa: F821  next grid slot

## Build the program

In [ ]:
F_READOUT = 2.75e9               # fixed readout frequency (Hz)
WIN = 40                         # demod integration window (batches)
IDLE = 200                       # idle head per shot — readout-path ring-down (batches)
SHOTS_PER_RUN = 1000             # 2*SHOTS_PER_RUN words <= the 16 KB (4096-word) core RAM

ro = ParamTable(1, F_READOUT, {"meas": Pulse(envelopes.square(WIN + 16), freq_hz=F_READOUT, amp=0.5)})
demod = ParamTable(2, 0.0, {"sq": Pulse(envelopes.square(WIN), amp=1.0)})

period = -(-(IDLE + LEAD + WIN + READOUT_LEAD) // 8) * 8            # fixed grid, multiple of 8

ro_code = units.freq_to_code(F_READOUT, m.params)                  # DAC-rate drive code
demod_code = units.demod_freq_to_code(F_READOUT, m.params)         # ADC-rate demod code (= 4*ro_code)

prog = compile_kernel(iq_shots, m, tables=dict(ro=ro, demod=demod),
                      out=Array(2 * SHOTS_PER_RUN), shots=SHOTS_PER_RUN, period=period,
                      ro_code=ro_code, demod_code=demod_code)

print(f"compiled: {SHOTS_PER_RUN} shots/run on a {period}-batch grid "
      f"({units.ns(period, m.params) / 1e3:.2f} us/shot), out = {prog.var_size('out') // 4} words")
print(f"readout drive code {ro_code} (ch 1), demod code {demod_code} (ch 2) @ {F_READOUT / 1e9:.2f} GHz")

## Collect 10,000 shots

`rq.setup` loads the image once (reset held between reruns); each `rq.rerun` reuses it and fires a
fresh 1,000 shots into `out`. Everything is fixed and baked into the program (`compile_kernel` folds the
scalar bindings in as constants), so the reruns take no params — only the shot noise differs.

In [ ]:
SHOTS = 10_000
runs = SHOTS // SHOTS_PER_RUN                          # 10 reruns of 1000 shots each

rq.setup(drv, m, {0: prog})                            # load image once; reset held between reruns
TIMEOUT = SHOTS_PER_RUN * period * 4 + 20_000_000      # poll budget: ~4 cycles/batch + boot slack

iq = np.empty(SHOTS, dtype=np.complex128)              # every shot's IQ, kept on the host
t0 = time.time()
for r in range(runs):                                  # shots/period/codes are baked into prog
    out = rq.rerun(drv, m, {0: prog}, results=["out"], timeout=TIMEOUT)[0]["out"]
    z = out.reshape(SHOTS_PER_RUN, 2)                  # (shot, [I, Q])
    iq[r * SHOTS_PER_RUN:(r + 1) * SHOTS_PER_RUN] = z[:, 0] + 1j * z[:, 1]
    print(f"  run {r + 1:>2}/{runs}: {SHOTS_PER_RUN} shots   ({time.time() - t0:.1f}s)")

print(f"done: {iq.size:,} IQ shots at {F_READOUT / 1e9:.2f} GHz in {runs} reruns, {time.time() - t0:.1f}s")

## Scatter plot

In [ ]:
I, Q = iq.real, iq.imag

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(I, Q, s=5, alpha=0.15, color="#1f77b4", edgecolors="none")
ax.scatter([I.mean()], [Q.mean()], color="#d62728", marker="+", s=200, lw=2,
           label=f"centroid ({I.mean():.0f}, {Q.mean():.0f})")
ax.set_aspect("equal")
ax.set_xlabel("I  (demod real)")
ax.set_ylabel("Q  (demod imag)")
ax.set_title(f"{iq.size:,} single-shot IQ acquisitions @ {F_READOUT / 1e9:.2f} GHz  ({m.params.name})")
ax.grid(alpha=0.3)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
plt.show()

snr = np.hypot(I.mean(), Q.mean()) / np.hypot(I.std(), Q.std())     # centroid / blob 1-sigma
print(f"centroid  I={I.mean():8.1f}  Q={Q.mean():8.1f}")
print(f"spread    dI={I.std():8.1f}  dQ={Q.std():8.1f}  (1-sigma)")
print(f"|IQ| mean={np.abs(iq).mean():.1f}   blob SNR ~ {snr:.1f}")

In [ ]:
drv.close()
print("disconnected")